# Chapter 05: Branch Protection & Rulesets (Reference)

## Learning Objectives

- Read a branch's light `protected` boolean with a token GITHUB_TOKEN-equivalent access has
- Attempt the full protection read and see why it needs Administration permission
- Extract required status check names from a full protection object
- Explain why Gate 1 is built around the light read specifically

## Setup

The next cell sets up reproducibility and the `PRA_MODE` toggle. You should see `PRA_MODE = 'fixture'` printed by default.

In [ ]:
import os
import random
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pr_automerge").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

RANDOM_STATE: int = 42
random.seed(RANDOM_STATE)

PRA_MODE = os.environ.get("PRA_MODE", "fixture")
PRA_REPO = os.environ.get("PRA_REPO", "")

if PRA_MODE == "live":
    assert PRA_REPO, "Set PRA_REPO=owner/name to run against a real repo"

print(f"PRA_MODE = {PRA_MODE!r}")


## 1. The Light Read

The next cell calls `check_protected_light`, hitting the endpoint that only needs `contents: read`. You should see `protected = True`.

In [ ]:
from labs.lab_05_branch_protection import check_protected_light

protected = check_protected_light("example/example")
print(f"protected = {protected}")


## 2. The Full Read

The next cell calls `fetch_protection_detail` and `required_check_names`. In fixture mode this succeeds (the fixture stands in for a PAT/App-token response); you should see the three required check names this repo actually registers: `test`, `gate2-pr-health`, `gate3-risk-score`.

In [ ]:
from labs.lab_05_branch_protection import fetch_protection_detail, required_check_names

detail = fetch_protection_detail("example/example")
checks = required_check_names(detail)
print(f"required status checks: {checks}")
print(f"enforce_admins: {detail.get('enforce_admins', {}).get('enabled')}")


## 3. Live Mode: The Real 403

The next cell only does something in `PRA_MODE=live`: it calls the full protection endpoint with the default GITHUB_TOKEN-equivalent auth GitHub Actions provides, which Chapter 05 Section 8 predicts will 403. In fixture mode it just explains what would happen.

In [ ]:
if PRA_MODE == "live":
    import subprocess
    r = subprocess.run(
        ["gh", "api", f"repos/{PRA_REPO}/branches/main/protection"],
        capture_output=True, text=True,
    )
    print(r.returncode, r.stderr.strip()[:200])
else:
    print("Fixture mode: this would 403 for GITHUB_TOKEN -- see Chapter 05 Section 8.")


## Takeaways & Next Steps

This notebook's takeaway is the contrast between the two reads above -- re-read the printed output before moving on.

In [ ]:
print("Re-run this notebook with PRA_MODE=live to see it against the real sandbox repo.")


---

📖 **Reading companion:** [Chapter 05: Branch Protection & Rulesets](../learning_modules/chapter_05_branch_protection.md)
